# 32 · A alta de churn de dezembro de 2025 para janeiro de 2026

Pergunta: o aumento de churn de dez/2025 para jan/2026 vem de um canal ou sistema específico?

Compara dezembro e janeiro por plano, canal de cobrança, sistema, canal de aquisição, tempo de casa
e tipo de saída, e repete a comparação um ano antes (dez/2024 para jan/2025) para separar
sazonalidade de novidade.

Engajamento: os treinos de `atividades_limpa` começam em 01/01/2026, então os níveis de
`src/engajamento.py` não podem ser calculados antes de 14/02/2026. Aqui o uso antes da saída vem de
`uso_mes` (treinos gravados no app no mês anterior), que existe para 2025.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
P = Path("../dados/processados"); S = Path("../saidas")

painel = pd.read_parquet(P / "assinaturas_mes.parquet", columns=["mes", "user_id", "plano", "canal_cobranca", "ativa_inicio_mes", "iniciou_no_mes", "perdida_no_mes", "motivo_fim"])
usuarios = pd.read_parquet(P / "usuarios_limpa.parquet", columns=["user_id", "sistema", "canal_aquisicao"])
assin = pd.read_parquet(P / "assinaturas_limpa.parquet", columns=["user_id", "inicio", "data_fim"])
resumo = pd.read_parquet(P / "assinaturas_resumo_eventos.parquet", columns=["user_id", "ultimo_desligamento"])
uso = pd.read_parquet(P / "uso_mes.parquet", columns=["user_id", "mes", "atividades_mes_anterior", "dias_ativos_mes_anterior", "origem_valor_mes_anterior"])

p = painel.merge(usuarios, on="user_id", how="left", validate="many_to_one").merge(assin, on="user_id", how="left", validate="many_to_one")
print("painel:", len(painel), "-> depois das junções:", len(p))
p = p[p.mes >= "2024-08-01"].copy()
print("sem jul/2024:", len(p))
p["casa"] = pd.cut((p.mes - p.inicio).dt.days, [-1, 60, 180, 365, 5000], labels=["até 2 meses", "2-6 meses", "6-12 meses", "12+ meses"])

def churn(d, por):
    g = d.groupby(por, observed=True).agg(ativas=("ativa_inicio_mes", "sum"), perdas=("perdida_no_mes", "sum"))
    g["churn %"] = g.perdas / g.ativas * 100
    return g

def dez_jan(d, por, dez, jan):
    a = churn(d[d.mes == dez], por); b = churn(d[d.mes == jan], por)
    t = pd.DataFrame({"ativas dez": a.ativas, "perdas dez": a.perdas, "churn dez %": a["churn %"],
                      "ativas jan": b.ativas, "perdas jan": b.perdas, "churn jan %": b["churn %"]})
    t["variação (p.p.)"] = t["churn jan %"] - t["churn dez %"]
    t["jan ÷ dez"] = t["churn jan %"] / t["churn dez %"]
    # perdas a mais em janeiro contra a taxa de dezembro, e a fatia de cada grupo no total
    t["perdas a mais"] = t["perdas jan"] - t["ativas jan"] * t["churn dez %"] / 100
    t["% das perdas a mais"] = t["perdas a mais"] / t["perdas a mais"].sum() * 100
    t["% da base jan"] = t["ativas jan"] / t["ativas jan"].sum() * 100
    return t

painel: 327771 -> depois das junções: 327771
sem jul/2024: 321927


## 1. A série: a alta de janeiro já aconteceu um ano antes

In [2]:
s = churn(p, ["mes", "plano"])
x = s["churn %"].unstack().join(s.perdas.unstack(), rsuffix=" (perdas)")
tot = churn(p, "mes"); x["total %"] = tot["churn %"]; x["total (perdas)"] = tot.perdas
print(x)
print()
for dez, jan in [("2024-12-01", "2025-01-01"), ("2025-12-01", "2026-01-01")]:
    print(f"== {dez[:7]} -> {jan[:7]}"); print(dez_jan(p, "plano", dez, jan)); print()

plano       anual  mensal  anual (perdas)  mensal (perdas)  total %  total (perdas)
mes                                                                                
2024-08-01    0.5     8.9              15              260      4.8             275
2024-09-01    0.8     9.1              25              295      5.0             320
2024-10-01    1.2     8.7              43              304      5.0             347
2024-11-01    1.2     7.8              44              298      4.5             342
2024-12-01    1.5     7.9              60              320      4.7             380
2025-01-01    2.0    10.7              88              446      6.3             534
2025-02-01    1.9     9.5              92              437      5.6             529
2025-03-01    1.8     9.5              91              461      5.5             552
2025-04-01    1.5     8.7              84              438      4.9             522
2025-05-01    1.7     8.6             100              443      5.0         

## 2. De onde vem a alta de dez/2025 para jan/2026

In [3]:
DEZ, JAN = "2025-12-01", "2026-01-01"
tabs = {}
for por in ["canal_cobranca", "sistema", "canal_aquisicao", "casa"]:
    for plano in ["mensal", "anual"]:
        t = dez_jan(p[p.plano == plano], por, DEZ, JAN)
        tabs[(plano, por)] = t
        print(f"== {plano.upper()} por {por}"); print(t.round(1)); print()
pd.concat(tabs, names=["plano", "dimensao", "valor"]).to_csv(S / "churn_dez25_jan26_por_dimensao.csv", encoding="utf-8-sig")

== MENSAL por canal_cobranca
                ativas dez  perdas dez  churn dez %  ativas jan  perdas jan  churn jan %  variação (p.p.)  jan ÷ dez  perdas a mais  % das perdas a mais  % da base jan
canal_cobranca                                                                                                                                                         
app_store             2065         156          7.6        2088         220         10.5              3.0        1.4           62.3                 33.4           34.7
google_play           1996         140          7.0        2016         242         12.0              5.0        1.7          100.6                 53.9           33.5
web_cartao            1203         122         10.1        1178         135         11.5              1.3        1.1           15.5                  8.3           19.6
web_pix                711          57          8.0         733          67          9.1              1.1        1.1            8.2

== ANUAL por canal_cobranca
                ativas dez  perdas dez  churn dez %  ativas jan  perdas jan  churn jan %  variação (p.p.)  jan ÷ dez  perdas a mais  % das perdas a mais  % da base jan
canal_cobranca                                                                                                                                                         
app_store             2453          72          2.9        2500         110          4.4              1.5        1.5           36.6                 50.4           35.1
google_play           2469          83          3.4        2483          90          3.6              0.3        1.1            6.5                  9.0           34.8
web_cartao            1541          44          2.9        1555          73          4.7              1.8        1.6           28.6                 39.4           21.8
web_pix                585          21          3.6         587          22          3.7              0.2        1.0            0.9 

== MENSAL por canal_aquisicao
                      ativas dez  perdas dez  churn dez %  ativas jan  perdas jan  churn jan %  variação (p.p.)  jan ÷ dez  perdas a mais  % das perdas a mais  % da base jan
canal_aquisicao                                                                                                                                                              
google_ads                   684          59          8.6         669          72         10.8              2.1        1.2           14.3                  7.7           11.1
indicacao                    960          75          7.8         956          99         10.4              2.5        1.3           24.3                 13.0           15.9
influenciador                791          77          9.7         823         118         14.3              4.6        1.5           37.9                 20.3           13.7
meta_ads                    1079         100          9.3        1052         136         12.9      

== MENSAL por casa
             ativas dez  perdas dez  churn dez %  ativas jan  perdas jan  churn jan %  variação (p.p.)  jan ÷ dez  perdas a mais  % das perdas a mais  % da base jan
casa                                                                                                                                                                
até 2 meses        1214          97          8.0        1054          88          8.3              0.4        1.0            3.8                  2.1           17.5
2-6 meses          1653         165         10.0        1788         251         14.0              4.1        1.4           72.5                 39.4           29.7
6-12 meses         1576         124          7.9        1595         201         12.6              4.7        1.6           75.5                 41.0           26.5
12+ meses          1532          89          5.8        1578         124          7.9              2.0        1.4           32.3                 17.6       

In [4]:
print("== mensal: canal de cobrança × sistema")
print(dez_jan(p[p.plano == "mensal"], ["sistema", "canal_cobranca"], DEZ, JAN).round(1))
print()
print("== por tipo de saída (% das ativas no início do mês)")
for plano in ["mensal", "anual"]:
    d = p[(p.plano == plano) & p.mes.isin(pd.to_datetime([DEZ, JAN]))]
    base = d.groupby("mes").ativa_inicio_mes.sum()
    n = pd.crosstab(d[d.perdida_no_mes].mes, d[d.perdida_no_mes].motivo_fim)
    print(plano); print(n); print((n.div(base, axis=0) * 100).round(2)); print()

== mensal: canal de cobrança × sistema
                        ativas dez  perdas dez  churn dez %  ativas jan  perdas jan  churn jan %  variação (p.p.)  jan ÷ dez  perdas a mais  % das perdas a mais  % da base jan
sistema canal_cobranca                                                                                                                                                         
android google_play           1996         140          7.0        2016         242         12.0              5.0        1.7          100.6                 53.9           33.5
        web_cartao             728          74         10.2         713          92         12.9              2.7        1.3           19.5                 10.5           11.9
        web_pix                403          33          8.2         414          42         10.1              2.0        1.2            8.1                  4.3            6.9
ios     app_store             2065         156          7.6        2088         2

anual
motivo_fim  chargeback  falha_pagamento  reembolso  renovacao_desligada
mes                                                                    
2025-12-01           0                2         10                  208
2026-01-01           1                6          1                  287
motivo_fim  chargeback  falha_pagamento  reembolso  renovacao_desligada
mes                                                                    
2025-12-01         0.0              0.0        0.1                  3.0
2026-01-01         0.0              0.1        0.0                  4.0



## 3. A mesma decomposição um ano antes (dez/2024 para jan/2025)

In [5]:
for por in ["canal_cobranca", "sistema", "canal_aquisicao", "casa"]:
    t = dez_jan(p[p.plano == "mensal"], por, "2024-12-01", "2025-01-01")
    print(f"== MENSAL por {por}"); print(t[["churn dez %", "churn jan %", "variação (p.p.)", "jan ÷ dez", "% das perdas a mais", "% da base jan"]].round(1)); print()

== MENSAL por canal_cobranca
                churn dez %  churn jan %  variação (p.p.)  jan ÷ dez  % das perdas a mais  % da base jan
canal_cobranca                                                                                          
app_store               8.3         10.3              2.1        1.2                 25.0           34.1
google_play             6.5         10.6              4.1        1.6                 48.8           33.6
web_cartao              7.8         11.2              3.4        1.4                 25.6           21.3
web_pix                10.7         10.8              0.2        1.0                  0.6           11.0

== MENSAL por sistema
         churn dez %  churn jan %  variação (p.p.)  jan ÷ dez  % das perdas a mais  % da base jan
sistema                                                                                          
android          7.6         10.6              3.0        1.4                 57.9           53.8
ios              8.2    

== MENSAL por canal_aquisicao
                      churn dez %  churn jan %  variação (p.p.)  jan ÷ dez  % das perdas a mais  % da base jan
canal_aquisicao                                                                                               
google_ads                    9.8          6.4             -3.4        0.7                -12.9           10.9
indicacao                     6.1          8.4              2.2        1.4                 13.0           16.3
influenciador                11.2         14.5              3.3        1.3                 15.5           13.3
meta_ads                      8.3         11.7              3.4        1.4                 21.3           17.5
organico                      6.8         10.6              3.8        1.6                 43.9           33.0
parceria_corporativa          6.4         12.4              6.0        1.9                 19.2            9.1



== MENSAL por casa
             churn dez %  churn jan %  variação (p.p.)  jan ÷ dez  % das perdas a mais  % da base jan
casa                                                                                                 
até 2 meses          8.1          8.2              0.2        1.0                  1.4           22.7
2-6 meses            9.5         12.0              2.5        1.3                 36.7           38.6
6-12 meses           8.0         14.3              6.3        1.8                 10.4            4.4
12+ meses            6.4         10.3              3.9        1.6                 51.5           34.3



## 4. Quando a saída de janeiro foi decidida e em que dia do mês acontece

In [6]:
l = p[p.perdida_no_mes & p.mes.isin(pd.to_datetime(["2025-11-01", DEZ, JAN, "2026-02-01"])) & (p.motivo_fim == "renovacao_desligada")].merge(resumo, on="user_id", how="left")
print("saídas voluntárias de nov/25 a fev/26:", len(l))
l["dias_antes"] = (l.data_fim - l.ultimo_desligamento.dt.normalize()).dt.days
l["mes_deslig"] = l.ultimo_desligamento.dt.to_period("M").astype(str)
for plano in ["mensal", "anual"]:
    d = l[l.plano == plano]
    print(plano, "- mês em que a renovação foi desligada (linhas = mês da saída), % da linha")
    t = pd.crosstab(d.mes.dt.to_period("M").astype(str), d.mes_deslig)
    t = t.loc[:, t.sum() >= 15]
    print((t.div(pd.crosstab(d.mes.dt.to_period("M").astype(str), d.mes_deslig).sum(axis=1), axis=0) * 100).round(0))
    print("dias do desligamento à saída (mediana):", d.groupby(d.mes.dt.to_period("M").astype(str)).dias_antes.median().to_dict()); print()
m = p[p.perdida_no_mes & (p.plano == "mensal") & p.mes.isin(pd.to_datetime([DEZ, JAN]))]
print("mensal: perdas por semana do mês")
print(pd.crosstab(pd.cut(m.data_fim.dt.day, [0, 7, 14, 21, 31], labels=["dias 1-7", "8-14", "15-21", "22-31"]), m.mes.dt.to_period("M").astype(str)))
print()
print("desligamentos da renovação por mês (todas as assinaturas, pelo último desligamento)")
r = resumo.merge(painel[["user_id", "plano"]].drop_duplicates(), on="user_id")
r = r[r.ultimo_desligamento.between("2025-09-01", "2026-03-31")]
print(pd.crosstab(r.ultimo_desligamento.dt.to_period("M").astype(str), r.plano))

saídas voluntárias de nov/25 a fev/26: 2935


mensal - mês em que a renovação foi desligada (linhas = mês da saída), % da linha
mes_deslig  2025-10  2025-11  2025-12  2026-01  2026-02
mes                                                    
2025-11        49.0     51.0      0.0      0.0      0.0
2025-12         0.0     47.0     53.0      0.0      0.0
2026-01         0.0      0.0     42.0     58.0      0.0
2026-02         0.0      0.0      0.0     54.0     46.0
dias do desligamento à saída (mediana): {'2025-11': 14.0, '2025-12': 15.0, '2026-01': 13.0, '2026-02': 14.0}

anual - mês em que a renovação foi desligada (linhas = mês da saída), % da linha
mes_deslig  2024-12  2025-01  2025-02  2025-03  2025-04  2025-05  2025-06  2025-07  2025-08  2025-09  2025-10  2025-11  2025-12  2026-01
mes                                                                                                                                     
2025-11         7.0     12.0      6.0      8.0      9.0     11.0      6.0     10.0      6.0     10.0     10.0      4.

plano                anual  mensal
ultimo_desligamento               
2025-09                228     436
2025-10                239     443
2025-11                278     428
2025-12                325     478
2026-01                391     638
2026-02                310     493
2026-03                299     770


## 5. Anual: quantas assinaturas fazem aniversário em cada mês

In [7]:
an = assin.merge(painel[["user_id", "plano"]].drop_duplicates(), on="user_id")
an = an[an.plano == "anual"]
for mes in ["2025-11", "2025-12", "2026-01", "2026-02"]:
    ini = pd.Period(mes, "M")
    k = an[(an.inicio.dt.month == ini.month) & (an.inicio.dt.year < ini.year) & (an.data_fim.isna() | (an.data_fim >= ini.start_time))]
    saiu = k.data_fim.between(ini.start_time, ini.end_time)
    print(mes, "| anuais com aniversário no mês:", len(k), "| saíram no mês:", int(saiu.sum()), f"({saiu.mean():.1%})")

2025-11 | anuais com aniversário no mês: 523 | saíram no mês: 211 (40.3%)
2025-12 | anuais com aniversário no mês: 469 | saíram no mês: 215 (45.8%)
2026-01 | anuais com aniversário no mês: 670 | saíram no mês: 291 (43.4%)
2026-02 | anuais com aniversário no mês: 572 | saíram no mês: 239 (41.8%)


## 6. Uso antes da saída (treinos gravados no app no mês anterior)

In [8]:
d = p[p.ativa_inicio_mes & p.mes.isin(pd.to_datetime([DEZ, JAN]))].merge(uso, on=["user_id", "mes"], how="left", validate="one_to_one")
print("assinatura-mês ativas em dez/25 e jan/26:", len(d), "| origem do uso do mês anterior:", d.origem_valor_mes_anterior.value_counts(dropna=False).to_dict())
d["treinos_ant"] = d.atividades_mes_anterior.astype("float")
d["faixa"] = pd.cut(d.treinos_ant, [-1, 0, 4, 9, 500], labels=["0 treinos", "1-4", "5-9", "10+"]).astype(object).fillna("desconhecido")
for plano in ["mensal", "anual"]:
    k = d[d.plano == plano]
    g = k.groupby(["faixa", k.mes.dt.to_period("M").astype(str)]).agg(ativas=("user_id", "size"), perdas=("perdida_no_mes", "sum"))
    g["churn %"] = g.perdas / g.ativas * 100
    print(plano, "- churn por treinos no mês anterior"); print(g.unstack().round(1)); print()
    print("% da base em cada faixa:"); print((pd.crosstab(k.faixa, k.mes.dt.to_period("M").astype(str), normalize="columns") * 100).round(1)); print()

assinatura-mês ativas em dez/25 e jan/26: 26163 | origem do uso do mês anterior: {'arquivo': 22454, 'zero_2025': 3709}
mensal - churn por treinos no mês anterior
           ativas          perdas         churn %        
mes       2025-12 2026-01 2025-12 2026-01 2025-12 2026-01
faixa                                                    
0 treinos     161     231      49      70    30.4    30.3
1-4          2439    2580     233     318     9.6    12.3
10+          1943    1611      79     130     4.1     8.1
5-9          1432    1593     114     146     8.0     9.2

% da base em cada faixa:
mes        2025-12  2026-01
faixa                      
0 treinos      2.7      3.8
1-4           40.8     42.9
10+           32.5     26.8
5-9           24.0     26.5

anual - churn por treinos no mês anterior
           ativas          perdas         churn %        
mes       2025-12 2026-01 2025-12 2026-01 2025-12 2026-01
faixa                                                    
0 treinos    1574    

mes        2025-12  2026-01
faixa                      
0 treinos     22.3     24.5
1-4           35.9     36.9
10+           24.3     19.4
5-9           17.5     19.3

